In [1]:
import requests 
import hashlib
import json
from collections import defaultdict

# 1. Descargamos usando la URL cruda (raw) directa
# docs_url = 'https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-intro/documents.json'
docs_url = 'https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2025/01-intro/documents.json'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()

documents = []
for course in documents_raw:
    course_name = course['course']
    for doc in course['documents']:
        doc['course'] = course_name
        documents.append(doc)

# 2. Función para generar el ID único (Hash)
def generate_document_id(doc):
    combined = f"{doc['course']}-{doc['question']}-{doc['text'][:10]}"
    hash_object = hashlib.md5(combined.encode())
    hash_hex = hash_object.hexdigest()
    document_id = hash_hex[:8]
    return document_id

# 3. Asignamos el ID a cada documento
for doc in documents:
    doc['id'] = generate_document_id(doc)

# 4. Verificamos si hay colisiones
hashes = defaultdict(list)
for doc in documents:
    doc_id = doc['id']
    hashes[doc_id].append(doc)

print(f"Total de IDs únicos: {len(hashes)}")
print(f"Total de documentos: {len(documents)}")

# 5. Guardamos nuestro nuevo dataset con IDs
with open('documents-with-ids.json', 'wt') as f_out:
    json.dump(documents, f_out, indent=2)

print("\n¡Archivo 'documents-with-ids.json' creado exitosamente!")
print(documents[3])

Total de IDs únicos: 947
Total de documentos: 948

¡Archivo 'documents-with-ids.json' creado exitosamente!
{'text': "You don't need it. You're accepted. You can also just start learning and submitting homework without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.", 'section': 'General course-related questions', 'question': 'Course - I have registered for the Data Engineering Bootcamp. When can I expect to receive the confirmation email?', 'course': 'data-engineering-zoomcamp', 'id': '0bbf41ec'}


In [2]:
from openai import OpenAI
import json

# 1. Inicializamos el cliente. (Asegúrate de que tu variable de entorno OPENAI_API_KEY esté configurada)
client = OpenAI()

# 2. Creamos la plantilla con las instrucciones para el LLM
prompt_template = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record. 

The record:

section: {section}
question: {question}
answer: {text}

Provide the output in parsable JSON without using code blocks:

["question1", "question2", ..., "question5"]
""".strip()

# 3. Función que envía el documento a OpenAI y nos devuelve las 5 preguntas
def generate_questions(doc):
    prompt = prompt_template.format(**doc)

    response = client.chat.completions.create(
        model='gpt-4o', # o gpt-4o-mini si quieres ahorrar costos
        messages=[{"role": "user", "content": prompt}]
    )

    json_response = response.choices[0].message.content
    return json_response

# 4. Hacemos una pequeña prueba con el documento número 3 para ver si funciona
print("Enviando prueba a OpenAI...")
prueba = generate_questions(documents[3])
print("\nResultado:")
print(prueba)

Enviando prueba a OpenAI...

Resultado:
["When will I get the confirmation email for the Data Engineering Bootcamp?", "Do I need to wait for a confirmation email before starting the course?", "Is registration necessary to begin submitting homework?", "What is the purpose of registering for the bootcamp?", "Can I start learning without being on the registered list?"]


In [4]:
import pickle
import pandas as pd
import os
import urllib.request

# 1. Descargamos el archivo 'results.bin' (El atajo del profesor para no gastar en OpenAI)
url_bin = 'https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2025/03-evaluation/search_evaluation/results.bin'
if not os.path.exists('results.bin'):
    print("Descargando results.bin...")
    urllib.request.urlretrieve(url_bin, 'results.bin')

# 2. Leemos el archivo usando la librería pickle
with open('results.bin', 'rb') as f_in:
    results = pickle.load(f_in)

# 3. Limpiamos las respuestas (Usamos try-except para ignorar las alucinaciones del LLM)
parsed_results = {}
for doc_id, json_questions in results.items():
    try:
        parsed_results[doc_id] = json.loads(json_questions)
    except json.JSONDecodeError:
        # Si OpenAI nos devolvió un JSON roto con un '\' extra, lo saltamos.
        print(f"Saltando el documento {doc_id} por formato JSON inválido.")
        continue

# 4. Construimos nuestro Ground Truth
doc_index = {d['id']: d for d in documents} 
final_results = []

for doc_id, questions in parsed_results.items():
    course = doc_index[doc_id]['course']
    for q in questions:
        final_results.append((q, course, doc_id))

# 5. Lo convertimos en una tabla y lo guardamos
df = pd.DataFrame(final_results, columns=['question', 'course', 'document'])
df.to_csv('ground-truth-data.csv', index=False)

print("\n¡Listo! El archivo 'ground-truth-data.csv' ha sido creado.")
print(f"Total de preguntas de evaluación válidas rescatadas: {len(df)}")
df.head()

Saltando el documento 58c9f99f por formato JSON inválido.

¡Listo! El archivo 'ground-truth-data.csv' ha sido creado.
Total de preguntas de evaluación válidas rescatadas: 4622


,question,course,document
0,When does the course begin?,data-engineering-zoomcamp,c02e79ef
1,How can I get the course schedule?,data-engineering-zoomcamp,c02e79ef
2,What is the link for course registration?,data-engineering-zoomcamp,c02e79ef
3,How can I receive course announcements?,data-engineering-zoomcamp,c02e79ef
4,Where do I join the Slack channel?,data-engineering-zoomcamp,c02e79ef
